In [1]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 260
BATCH_SIZE = 16
ABLATION_EPOCHS = 20          # updated from 10 -> matches current tuned training budget
NUM_CLASSES = 3
NUM_WORKERS = 0
SEED = 42

CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
DATA_ROOT = PROJECT_ROOT / "dataset" / "processed_dataset"
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "validation"
ABLATION_RESULTS_PATH = PROJECT_ROOT / "results" / "ablation_results_20ep.csv"
ABLATION_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [3]:
# ============================================================
# Cell 3: Data Preparation
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_transform)

assert train_dataset.classes == CLASS_NAMES, f"Class order mismatch: {train_dataset.classes}"

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

train_labels = [label for _, label in train_dataset.samples]
class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Train samples:", len(train_dataset), "| Val samples:", len(val_dataset))
print("Class weights:", class_weights)

Train samples: 4099 | Val samples: 878
Class weights: [0.70212402 1.23315283 1.30749601]


In [ ]:
# ============================================================
# Cell 4: Building Blocks (same definitions as Notebook 09)
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention()
    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

class MultiScaleFeatureFusion(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        mid = in_channels // reduction
        self.reduce = nn.Sequential(nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_1x1 = nn.Sequential(nn.Conv2d(mid, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_3x3 = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.branch_5x5 = nn.Sequential(nn.Conv2d(mid, mid, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))
        self.fusion = nn.Sequential(nn.Conv2d(mid * 3, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels), nn.ReLU(inplace=True))
    def forward(self, x):
        x = self.reduce(x)
        f = torch.cat([self.branch_1x1(x), self.branch_3x3(x), self.branch_5x5(x)], dim=1)
        return self.fusion(f)

class AdaptiveFeatureFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight_generator = nn.Sequential(
            nn.Conv2d(channels * 2, channels, 1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 1, bias=False), nn.Sigmoid()
        )
    def forward(self, feature_a, feature_b):
        weights = self.weight_generator(torch.cat([feature_a, feature_b], dim=1))
        return weights * feature_a + (1.0 - weights) * feature_b

class ResidualEnhancement(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.refine = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True))
    def forward(self, x):
        return x + self.refine(x)

Cell 4 : Building blocks defined.


In [5]:
# ============================================================
# Cell 5: Configurable PneumoXNet for Ablation
# ============================================================

class PneumoXNetAblation(nn.Module):
    """
    use_cbam, use_multiscale, use_aff, use_residual control which
    modules are active. When a branch is disabled, raw backbone
    features are used in its place so the pipeline shape stays valid.
    """
    def __init__(self, num_classes=3, use_cbam=True, use_multiscale=True,
                 use_aff=True, use_residual=True):
        super().__init__()

        self.use_cbam = use_cbam
        self.use_multiscale = use_multiscale
        self.use_aff = use_aff
        self.use_residual = use_residual

        weights = EfficientNet_B2_Weights.DEFAULT
        backbone = efficientnet_b2(weights=weights)
        self.backbone = backbone.features
        self.feature_channels = 1408

        self.cbam = CBAM(self.feature_channels) if use_cbam else None
        self.multiscale = MultiScaleFeatureFusion(self.feature_channels) if use_multiscale else None
        self.aff = AdaptiveFeatureFusion(self.feature_channels) if use_aff else None
        self.residual = ResidualEnhancement(self.feature_channels) if use_residual else None

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(self.feature_channels, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        branch_a = self.cbam(features) if self.use_cbam else features
        branch_b = self.multiscale(features) if self.use_multiscale else features

        if self.use_aff:
            fused = self.aff(branch_a, branch_b)
        else:
            fused = (branch_a + branch_b) / 2.0    # simple average fallback

        enhanced = self.residual(fused) if self.use_residual else fused

        pooled = self.pool(enhanced)
        return self.classifier(pooled)


print("Cell 5 : Configurable PneumoXNetAblation defined.")

Cell 5 : Configurable PneumoXNetAblation defined.


In [6]:
# ============================================================
# Cell 6: Train / Validate Functions (lightweight, no scheduler)
# ============================================================

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / total, correct / total, macro_f1


print("Cell 6 : Train/Validate functions ready.")

Cell 6 : Train/Validate functions ready.


In [7]:
# ============================================================
# Cell 7: Run Ablation Study
# ============================================================

ablation_configs = {
    "Full PneumoXNet":        dict(use_cbam=True,  use_multiscale=True,  use_aff=True,  use_residual=True),
    "w/o CBAM":                dict(use_cbam=False, use_multiscale=True,  use_aff=True,  use_residual=True),
    "w/o MultiScaleFusion":    dict(use_cbam=True,  use_multiscale=False, use_aff=True,  use_residual=True),
    "w/o AdaptiveFusion":      dict(use_cbam=True,  use_multiscale=True,  use_aff=False, use_residual=True),
    "w/o ResidualEnhancement": dict(use_cbam=True,  use_multiscale=True,  use_aff=True,  use_residual=False),
}

ablation_results = []

for variant_name, flags in ablation_configs.items():

    print("=" * 70)
    print(f"Training Variant : {variant_name}")
    print("=" * 70)

    model = PneumoXNetAblation(num_classes=NUM_CLASSES, **flags).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=2e-4)

    best_val_acc = 0.0
    best_val_f1 = 0.0
    start_time = time.time()

    for epoch in range(ABLATION_EPOCHS):

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_f1 = validate_one_epoch(model, valid_loader, criterion, DEVICE)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_f1 = val_f1

        print(f"Epoch [{epoch+1}/{ABLATION_EPOCHS}] "
              f"Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}  Val F1: {val_f1:.4f}")

    elapsed = (time.time() - start_time) / 60

    ablation_results.append({
        "Variant": variant_name,
        "Best Valid Accuracy": round(best_val_acc, 4),
        "Best Valid Macro-F1": round(best_val_f1, 4),
        "Training Time (min)": round(elapsed, 2),
        "Total Params": sum(p.numel() for p in model.parameters())
    })

    del model
    torch.cuda.empty_cache()

    print(f"\n{variant_name} done in {elapsed:.2f} min. Best Val Acc: {best_val_acc:.4f}\n")

print("\nAblation study completed for all variants.")

Training Variant : Full PneumoXNet
Epoch [1/10] Train Acc: 0.7292  Val Acc: 0.7893  Val F1: 0.7848
Epoch [2/10] Train Acc: 0.7826  Val Acc: 0.7745  Val F1: 0.7791
Epoch [3/10] Train Acc: 0.7987  Val Acc: 0.7380  Val F1: 0.7509
Epoch [4/10] Train Acc: 0.8290  Val Acc: 0.7882  Val F1: 0.7940
Epoch [5/10] Train Acc: 0.8309  Val Acc: 0.7472  Val F1: 0.7628
Epoch [6/10] Train Acc: 0.8522  Val Acc: 0.7961  Val F1: 0.8001
Epoch [7/10] Train Acc: 0.8524  Val Acc: 0.7722  Val F1: 0.7853
Epoch [8/10] Train Acc: 0.8692  Val Acc: 0.8257  Val F1: 0.8303
Epoch [9/10] Train Acc: 0.8851  Val Acc: 0.8360  Val F1: 0.8374
Epoch [10/10] Train Acc: 0.8944  Val Acc: 0.8064  Val F1: 0.8147

Full PneumoXNet done in 25.29 min. Best Val Acc: 0.8360

Training Variant : w/o CBAM
Epoch [1/10] Train Acc: 0.7326  Val Acc: 0.7836  Val F1: 0.7870
Epoch [2/10] Train Acc: 0.7799  Val Acc: 0.7756  Val F1: 0.7826
Epoch [3/10] Train Acc: 0.8134  Val Acc: 0.7927  Val F1: 0.7980
Epoch [4/10] Train Acc: 0.8278  Val Acc: 0.791

In [8]:
# ============================================================
# Cell 8: Ablation Results Summary
# ============================================================

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(ABLATION_RESULTS_PATH, index=False)

print("=" * 70)
print("Ablation Study — Final Results")
print("=" * 70)
print(ablation_df.to_string(index=False))
print("=" * 70)
print(f"Saved to : {ABLATION_RESULTS_PATH}")

Ablation Study — Final Results
                Variant  Best Valid Accuracy  Best Valid Macro-F1  Training Time (min)  Total Params
        Full PneumoXNet               0.8360               0.8374                25.29      36809319
               w/o CBAM               0.8166               0.8197                23.65      36561413
   w/o MultiScaleFusion               0.8326               0.8351                23.13      32467047
     w/o AdaptiveFusion               0.8485               0.8426                22.92      30859111
w/o ResidualEnhancement               0.8337               0.8341                22.42      18964327
Saved to : /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/results/ablation_study_results.csv
